# Confronto Robyn vs Meridian - Parte 2: Meridian sullo stesso dataset

Fit di **Google Meridian** (modello *national*, singola geo) su
`dt_simulated_weekly`, lo stesso dataset demo di Robyn.

**Prima di iniziare:** Runtime -> Cambia tipo di runtime -> **GPU (T4)**.
Tempo stimato: ~15-30 min di campionamento MCMC.

Allineamento col notebook Robyn (vedi `protocollo_confronto.md`):
- stessi dati, stessa finestra (tutte le 208 settimane)
- **holdout = ultime ~21 settimane (10%)** = finestra test di Robyn
- stesse variabili: 5 canali paid (esposizione dove disponibile),
  newsletter come media organico, competitor_sales_B + events come controlli

Output: `meridian_metrics.csv` (R2/MAPE/wMAPE train-test), `meridian_roas.csv`
(ROAS per canale con intervalli di credibilita'), grafici di fit.

## 1. Setup

In [ ]:
!pip install -q google-meridian pyreadr
import tensorflow as tf
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Dataset demo di Robyn

Scaricato direttamente dal repository ufficiale di Robyn (file `.RData`).
Se il download fallisce, decommenta la cella di upload e carica il
`dt_simulated_weekly.csv` esportato dal notebook Robyn.

In [ ]:
import urllib.request
import numpy as np
import pandas as pd
import pyreadr

URL = "https://github.com/facebookexperimental/Robyn/raw/main/R/data/dt_simulated_weekly.RData"
urllib.request.urlretrieve(URL, "dt_simulated_weekly.RData")
df = pyreadr.read_r("dt_simulated_weekly.RData")["dt_simulated_weekly"]

# --- alternativa: upload del CSV esportato dal notebook Robyn ---
# from google.colab import files
# files.upload()  # scegli dt_simulated_weekly.csv
# df = pd.read_csv("dt_simulated_weekly.csv")

print(df.shape)
df.head()

## 3. Preparazione dati

- `events` (categorica) -> dummy 0/1 tra i controlli
- geo fittizia unica: con una sola geo Meridian stima il modello *national*

In [ ]:
df = df.sort_values("DATE").reset_index(drop=True)
df["time"] = pd.to_datetime(df["DATE"]).dt.strftime("%Y-%m-%d")
df["geo"] = "national"

ev = pd.get_dummies(df["events"].astype(str), prefix="evento", dtype=float)
ev = ev.drop(columns=[c for c in ev.columns if c.lower().endswith("_na")],
             errors="ignore")
df = pd.concat([df, ev], axis=1)

controls = ["competitor_sales_B"] + list(ev.columns)
channels = ["tv", "ooh", "print", "facebook", "search"]
media_cols = ["tv_S", "ooh_S", "print_S", "facebook_I", "search_clicks_P"]
spend_cols = ["tv_S", "ooh_S", "print_S", "facebook_S", "search_S"]

n_times = df["time"].nunique()
print(f"{n_times} settimane, controlli: {controls}")

## 4. InputData + ModelSpec

- `holdout_id`: ultime ~21 settimane escluse dal training del KPI ->
  metriche out-of-sample confrontabili col test di Robyn. (Meridian
  raccomanda un holdout bilanciato nel tempo; qui si usa una finestra
  finale per comparabilita' con Robyn -- scelta da dichiarare in tesi.)
- `knots` ~ 1 al mese: trend + stagionalita' via intercetta variabile
  (l'equivalente funzionale di prophet in Robyn)
- prior di default (nessuna calibrazione, come Robyn senza esperimenti)

In [ ]:
from meridian.data import data_frame_input_data_builder as B
from meridian.model import model as M
from meridian.model import prior_distribution as P
from meridian.model import spec as S

builder = (B.DataFrameInputDataBuilder(kpi_type="revenue",
                                       default_geo_column="geo")
           .with_kpi(df, kpi_col="revenue")
           .with_controls(df, control_cols=controls)
           .with_media(df,
                       media_cols=media_cols,
                       media_spend_cols=spend_cols,
                       media_channels=channels)
           .with_organic_media(df,
                               organic_media_cols=["newsletter"],
                               organic_media_channels=["newsletter"]))
data = builder.build()

n_test = int(round(n_times * 0.10))            # ~21 settimane
holdout = np.zeros(n_times, dtype=bool)
holdout[-n_test:] = True
print(f"Holdout: ultime {n_test} settimane")

knots = int(np.ceil(n_times / 4))              # ~1 knot al mese
mspec = S.ModelSpec(prior=P.PriorDistribution(),
                    knots=knots,
                    max_lag=8,
                    paid_media_prior_type="roi",
                    holdout_id=holdout)
mmm = M.Meridian(input_data=data, model_spec=mspec)

## 5. Fit MCMC (~15-30 min su T4)

In [ ]:
mmm.sample_prior(500, seed=42)
mmm.sample_posterior(n_chains=4, n_adapt=500, n_burnin=500, n_keep=1000,
                     seed=42)

## 6. Diagnostica e accuratezza predittiva

R-hat < 1.05 su tutti i parametri = catene convergenti (prerequisito prima
di leggere qualsiasi metrica). `predictive_accuracy` restituisce R2, MAPE e
wMAPE separati per Train e Test: il **Test** e' la riga da confrontare con
`rsq_test` / `nrmse_test` di Robyn.

In [ ]:
from meridian.analysis import analyzer as A
from meridian.analysis import visualizer as V

V.ModelDiagnostics(mmm).plot_rhat_boxplot()

In [ ]:
an = A.Analyzer(mmm)
acc = an.predictive_accuracy()
acc_df = acc.to_dataframe().reset_index()
acc_df.to_csv("meridian_metrics.csv", index=False)
acc_df

In [ ]:
# fit atteso vs osservato (il tratto finale e' l'holdout)
V.ModelFit(mmm).plot_model_fit()

## 7. ROAS per canale (posterior)

A differenza di Robyn (stima puntuale per soluzione), qui ogni ROAS ha una
distribuzione: si esportano media e intervallo di credibilita' al 90%.

In [ ]:
roi = np.asarray(an.roi())   # (chains, draws, canali) nell'ordine di `channels`
roas = pd.DataFrame({
    "canale": channels,
    "roi_medio": roi.mean(axis=(0, 1)),
    "roi_q05": np.quantile(roi, 0.05, axis=(0, 1)),
    "roi_q50": np.quantile(roi, 0.50, axis=(0, 1)),
    "roi_q95": np.quantile(roi, 0.95, axis=(0, 1)),
})
roas.to_csv("meridian_roas.csv", index=False)
roas

In [ ]:
# curve di risposta (confronto qualitativo con gli onepager di Robyn)
V.MediaEffects(mmm).plot_response_curves()

## 8. Scarica i risultati

In [ ]:
from google.colab import files
import shutil, os
os.makedirs("meridian_output", exist_ok=True)
for f in ("meridian_metrics.csv", "meridian_roas.csv"):
    shutil.copy(f, "meridian_output/")
shutil.make_archive("meridian_risultati", "zip", "meridian_output")
files.download("meridian_risultati.zip")

### (opzionale) Stabilita' tra run
Rilancia le sezioni 5-7 con `seed=43` e `seed=44`: con MCMC convergente i
ROAS dovrebbero variare molto poco tra i run (punto di forza da confrontare
con la variabilita' tra soluzioni di Robyn).